In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
%matplotlib inline

In [2]:

words = open('turkce_isimler.txt', 'r', encoding='utf-8').read().splitlines()
print(f"Toplam isim{len(words)}")
print( words[:8])


chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

print(stoi)

Toplam isim13872
['aba', 'abaca', 'abacan', 'abaç', 'abay', 'abayhan', 'abaza', 'abbas']
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, 'ç': 26, 'ö': 27, 'ü': 28, 'ğ': 29, 'ı': 30, 'ş': 31, '.': 0}


In [3]:

N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    N[ix1, ix2] += 1

# Model smoothing
P = (N + 1).float()
P /= P.sum(1, keepdim=True)

# Bigram Loss
nll = 0.0
n = 0
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    nll -= torch.log(prob)
    n += 1

bigram_loss = (nll / n).item()
print(f"Bigram NLL loss: {bigram_loss:.4f}")

Bigram NLL loss: 2.5005


In [4]:

g_bg = torch.Generator().manual_seed(2147483647)

bigram_names = []
for _ in range(10):
  out = []
  ix = 0
  while True:
    p = P[ix]
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g_bg).item()
    out.append(itos[ix])
    if ix == 0:
      break
  bigram_names.append(''.join(out[:-1]))

print("Bigram modeli:\n---------------------------")
for name in bigram_names:
  print(name)

Bigram modeli:
---------------------------
münide
ilkarah
p
suraye
ın
fehin
ur
taşınure
bdinerave
bileneyikdedainsuze


In [5]:
block_size = 3

def build_dataset(words_list):
  X, Y = [], []
  for w in words_list:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

print(f"Train: {Xtr.shape}")
print(f"Dev:   {Xdev.shape}")
print(f"Test:  {Xte.shape}")

Train: torch.Size([80060, 3])
Dev:   torch.Size([9906, 3])
Test:  torch.Size([9965, 3])


In [6]:
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * (5/3) / ((block_size * n_embd)**0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0

parameters = [C, W1, b1, W2, b2]
print("Toplam Parametre Sayısı:", sum(p.nelement() for p in parameters))

for p in parameters:
  p.requires_grad = True

Toplam Parametre Sayısı: 12952


In [7]:
max_steps = 30000
batch_size = 32
lossi = []

for i in range(max_steps):
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix]
  #forward pass
  emb = C[Xb] 
  h = torch.tanh(emb.view(-1, block_size * n_embd) @ W1 + b1)
  logits = h @ W2 + b2 
  loss = F.cross_entropy(logits, Yb)
  
  # Backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # Update
  lr = 0.1 if i < 20000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad
    
  if i % 10000 == 0:
    print(f"{i:7d}/{max_steps:7d}: {loss.item():.4f}")
  lossi.append(loss.log10().item())

      0/  30000: 3.4631
  10000/  30000: 2.1130
  20000/  30000: 1.9610


In [8]:

with torch.no_grad():
  emb_tr = C[Xtr]
  h_tr = torch.tanh(emb_tr.view(-1, block_size * n_embd) @ W1 + b1)
  logits_tr = h_tr @ W2 + b2
  loss_train = F.cross_entropy(logits_tr, Ytr).item()

  emb_dev = C[Xdev]
  h_dev = torch.tanh(emb_dev.view(-1, block_size * n_embd) @ W1 + b1)
  logits_dev = h_dev @ W2 + b2
  loss_dev = F.cross_entropy(logits_dev, Ydev).item()

print(f"MLP train loss: {loss_train:.4f}")
print(f"MLP dev train loss:  {loss_dev:.4f}")

MLP train loss: 2.0373
MLP dev train loss:  2.1173


In [9]:

g_sample = torch.Generator().manual_seed(2147483647 + 5)

mlp_names = []
for _ in range(10):
  out = []
  context = [0] * block_size
  while True:
    emb = C[torch.tensor([context])]
    h = torch.tanh(emb.view(1, -1) @ W1 + b1)
    logits = h @ W2 + b2
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1, generator=g_sample).item()
    context = context[1:] + [ix]
    out.append(ix)
    if ix == 0:
      break
  mlp_names.append(''.join(itos[i] for i in out[:-1]))

print("MLP modeli\n-----------------------------")
for name in mlp_names:
  print(name)

MLP modeli
-----------------------------
bahi
dümengüce
melim
aybay
can
ten
nasırap
gülmerengunnanabahazemz
kaprina
sadin


### Bigram vs. MLP

In [10]:

print(f"{'Özellik':<25} | {'Bigram (Part 1)':<25} | {'MLP (Part 2/3)':<25}")
print("-" * 80)
print(f"{'Dev / Test Loss':<25} | {bigram_loss:<25.4f} | {loss_dev:<25.4f}")
print("-" * 80)
print(f"{'Üretilen Örnekler':<25} | {'Bigram İsimleri':<25} | {'MLP İsimleri':<25}")
print("-" * 80)

for i in range(10):
  bg_n = bigram_names[i] if i < len(bigram_names) else ''
  mlp_n = mlp_names[i] if i < len(mlp_names) else ''
  print(f"Örnek {i+1:<19} | {bg_n:<25} | {mlp_n:<25}")

Özellik                   | Bigram (Part 1)           | MLP (Part 2/3)           
--------------------------------------------------------------------------------
Dev / Test Loss           | 2.5005                    | 2.1173                   
--------------------------------------------------------------------------------
Üretilen Örnekler         | Bigram İsimleri           | MLP İsimleri             
--------------------------------------------------------------------------------
Örnek 1                   | münide                    | bahi                     
Örnek 2                   | ilkarah                   | dümengüce                
Örnek 3                   | p                         | melim                    
Örnek 4                   | suraye                    | aybay                    
Örnek 5                   | ın                        | can                      
Örnek 6                   | fehin                     | ten                      
Örnek 7            